# Zero-to-Hero: Production Open-Source LLM Fine-Tuning Framework

This notebook walks through architecture, concepts, and runnable pipeline steps for Project #22.

## 1. LLMs and Transformers

Large Language Models (LLMs) are autoregressive sequence models trained to predict next tokens. Transformer architecture enables long-context reasoning via self-attention, where token representations attend across sequence positions.

In production, model quality is only one part of the system. Data quality, reproducibility, evaluation reliability, safety controls, and deployment latency define actual business value.

## 2. Instruction Tuning and SFT

Instruction tuning adapts a base model to follow task instructions. Supervised Fine-Tuning (SFT) trains on `(instruction, context, response)` tuples.

This framework supports multiple prompt templates (Alpaca, ChatML, Llama 3, Mistral, Qwen, Phi, Gemma, Custom) to match model chat formats.

## 3. LoRA, QLoRA, and PEFT

PEFT (Parameter-Efficient Fine-Tuning) updates low-rank adapters instead of full model weights.

- **LoRA**: train rank-decomposed updates.
- **QLoRA**: quantized base weights + LoRA adapters for low VRAM training.
- **RSLoRA/DoRA**: additional variants for robustness and adaptation behavior.

Framework design includes adapter management, merging manifests, and stack manifests for reproducible adapter workflows.

## 4. Quantization and Inference

Quantization lowers memory/latency tradeoffs. This project exposes 4-bit, 8-bit, FP16, BF16 configuration and export contracts for GGUF/Ollama/vLLM.

Inference backends in this framework:
- Transformers local runtime
- vLLM HTTP endpoint
- Ollama generate API

## 5. Data Pipeline

Pipeline responsibilities:
- dataset ingestion (download/stream hooks)
- filtering and deduplication
- prompt formatting
- token/length profiling
- train/validation split
- version manifest generation

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from llmft.pipeline import ProjectRunner

runner = ProjectRunner.from_config_path('configs/project.yaml')
runner.config.runtime.artifacts_dir = 'artifacts/notebook_run'
runner.config.runtime.cache_dir = '.cache/notebook_run'
runner.config.inference.vllm_host = 'http://127.0.0.1:9'
runner.config.inference.ollama_host = 'http://127.0.0.1:9'
runner

## 6. Environment Validation and Dataset Build

In [ ]:
env_report = runner.validate_env()
bundle = runner.build_data()
str(env_report), bundle.version_id, bundle.stats

## 7. Training + Hyperparameter Optimization

In constrained environments this notebook uses dry-run mode; framework still emits reproducible training/HPO artifacts and metadata contracts.

In [ ]:
train_report = runner.train_sft(dry_run=True)
train_report

## 8. Evaluation and LLM-as-Judge Report

Evaluation produces metric outputs and structured judge scores for correctness/helpfulness/faithfulness/coherence/instruction-following/grounding/safety.

In [ ]:
eval_report = runner.run_evaluation()
eval_report

## 9. Inference Backends and Benchmarking

Runs latency benchmark across Transformers, vLLM, and Ollama contracts.

In [ ]:
bench_report = runner.run_benchmark()
infer_report = runner.run_inference('Explain why QLoRA helps low-VRAM tuning.')
bench_report, infer_report

## 10. Export and Deployment Artifacts

Export stage emits adapter bundle, merged manifest, GGUF placeholder, and Ollama Modelfile for downstream deployment workflows.

In [ ]:
export_report = runner.run_export()
telemetry_report = runner.telemetry_snapshot()
export_report, telemetry_report

## 11. Safety, Observability, and Production Notes

Production-grade systems need safety gates and telemetry, not only model quality:
- prompt injection checks
- dataset sanitization
- unsafe response detection
- runtime snapshots + experiment tracking

When full GPU stack is available, switch to real training paths and keep same orchestration contracts for reproducibility.